# Feature Engineering en SQL

A continuación, veremos cómo calcular diferentes variables para el feature engineering utilizando SQL.


In [ ]:
#%pip install jupysql
#%pip install duckdb-engine

In [4]:
import duckdb
import pandas as pd

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

%sql duckdb://

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [5]:
base_path = "/Users/carof/Downloads/DENICOLAY/"
dataset_path = base_path + 'data/' 
dataset_file = 'competencia_01.csv'


In [6]:
%%sql
create or replace table competencia_01 as
select
    *
from read_csv_auto("{{dataset_path + dataset_file}}")

,Success


### Variables intra mes

1. Features Intra-Mes (Foto Actual)

Estas variables capturan ratios de intensidad, bancarización, riesgo y uso del crédito dentro del mismo mes $t$:

Agregaciones Totales de Tarjetas de Crédito:

mconsumototal_tc = mtarjeta_visa_consumo + mtarjeta_master_consumocconsumos_total_tc = ctarjeta_visa_transacciones + ctarjeta_master_transaccionesmsaldototal_tc = Visa_msaldototal + Master_msaldototalmlimitecompra_total = Visa_mlimitecompra + Master_mlimitecomprampagominimo_total = Visa_mpagominimo + Master_mpagominimompagado_total = Visa_mpagado + Master_mpagado

Ratios de Utilización y Apalancamiento:

ratio_utilizacion_limite = msaldototal_tc / (mlimitecompra_total + 0.01) (señales de saturación crediticia o desuso)ratio_pago_vs_saldo = mpagado_total / (msaldototal_tc + 0.01)ratio_pago_minimo = mpagado_total / (mpagominimo_total + 0.01) (si paga exactamente el mínimo o menos, hay estrés financiero)

Patrimonio Líquido y Pasivos:

saldo_liquido_total = mcuentas_saldo + mplazo_fijo_pesos + minversion1_pesos + minversion2deuda_prestamos_total = mprestamos_personales + mprestamos_prendarios + mprestamos_hipotecariosratio_liquidez_vs_deuda = saldo_liquido_total / (deuda_prestamos_total + 0.01)

Intensidad Digital vs. Físico:

trx_digitales = chomebanking_transacciones + cmobile_app_trxtrx_fisicas = ccajas_transacciones + catm_trx + ccallcenter_transaccionesratio_canal_digital = trx_digitales / (trx_digitales + trx_fisicas + 0.01)

Ticket Promedio Transaccional:

ticket_promedio_visa = mtarjeta_visa_consumo / (ctarjeta_visa_transacciones + 0.01)ticket_promedio_master = mtarjeta_master_consumo / (ctarjeta_master_transacciones + 0.01)ticket_promedio_debito = mautoservicio / (ctarjeta_debito_transacciones + 0.01)

Diferencia Temporal en Tarjetas:

Antigüedad de la tarjeta respecto a la foto actual: antiguedad_dias_visa = foto_mes - Visa_fechaalta (convirtiendo a días o meses).Días desde el último vencimiento / inicio de mora.

In [7]:
%%sql
CREATE OR REPLACE TABLE competencia_01_fe AS
WITH base_agregaciones AS (
    SELECT
        *,

        -- 1. Agregaciones totales de Tarjetas de Crédito (Visa + Master)
        (COALESCE(mtarjeta_visa_consumo, 0) + COALESCE(mtarjeta_master_consumo, 0)) AS mconsumototal_tc,
        (COALESCE(ctarjeta_visa_transacciones, 0) + COALESCE(ctarjeta_master_transacciones, 0)) AS cconsumos_total_tc,
        (COALESCE(Visa_msaldototal, 0) + COALESCE(Master_msaldototal, 0)) AS msaldototal_tc,
        (COALESCE(Visa_mlimitecompra, 0) + COALESCE(Master_mlimitecompra, 0)) AS mlimitecompra_total,
        (COALESCE(Visa_mpagominimo, 0) + COALESCE(Master_mpagominimo, 0)) AS mpagominimo_total,
        (COALESCE(Visa_mpagado, 0) + COALESCE(Master_mpagado, 0)) AS mpagado_total,

        -- 2. Patrimonio Líquido y Deuda en Préstamos
        (COALESCE(mcuentas_saldo, 0) + COALESCE(mplazo_fijo_pesos, 0) + 
         COALESCE(minversion1_pesos, 0) + COALESCE(minversion2, 0)) AS saldo_liquido_total,
        (COALESCE(mprestamos_personales, 0) + COALESCE(mprestamos_prendarios, 0) + 
         COALESCE(mprestamos_hipotecarios, 0)) AS deuda_prestamos_total,

        -- 3. Canales Digitales vs Físicos
        (COALESCE(chomebanking_transacciones, 0) + COALESCE(cmobile_app_trx, 0)) AS trx_digitales,
        (COALESCE(ccajas_transacciones, 0) + COALESCE(catm_trx, 0) + COALESCE(ccallcenter_transacciones, 0)) AS trx_fisicas,

        -- 4. Normalización de fechas para cálculo de antigüedad
        strptime(CAST(foto_mes AS VARCHAR) || '01', '%Y%m%d')::DATE AS fecha_foto,
        CASE 
            WHEN Visa_fechaalta IS NOT NULL AND Visa_fechaalta > 0 
            THEN try_strptime(CAST(Visa_fechaalta AS VARCHAR), '%Y%m%d')::DATE 
            ELSE NULL 
        END AS visa_fechaalta_date,
        CASE 
            WHEN Master_fechaalta IS NOT NULL AND Master_fechaalta > 0 
            THEN try_strptime(CAST(Master_fechaalta AS VARCHAR), '%Y%m%d')::DATE 
            ELSE NULL 
        END AS master_fechaalta_date

    FROM competencia_01
)
SELECT
    * EXCLUDE (fecha_foto, visa_fechaalta_date, master_fechaalta_date),

    -- Ratios de Tarjetas de Crédito
    ROUND(msaldototal_tc / NULLIF(mlimitecompra_total, 0), 4) AS ratio_utilizacion_limite,
    ROUND(mpagado_total / NULLIF(msaldototal_tc, 0), 4) AS ratio_pago_vs_saldo,
    ROUND(mpagado_total / NULLIF(mpagominimo_total, 0), 4) AS ratio_pago_minimo,

    -- Solvencia y Liquidez
    ROUND(saldo_liquido_total / NULLIF(deuda_prestamos_total, 0), 4) AS ratio_liquidez_vs_deuda,

    -- Intensidad Digital
    ROUND(trx_digitales / NULLIF(trx_digitales + trx_fisicas, 0), 4) AS ratio_canal_digital,

    -- Tickets Promedio
    ROUND(mtarjeta_visa_consumo / NULLIF(ctarjeta_visa_transacciones, 0), 2) AS ticket_promedio_visa,
    ROUND(mtarjeta_master_consumo / NULLIF(ctarjeta_master_transacciones, 0), 2) AS ticket_promedio_master,
    ROUND(mautoservicio / NULLIF(ctarjeta_debito_transacciones, 0), 2) AS ticket_promedio_debito,

    -- Antigüedades en meses de tarjetas
    DATE_DIFF('month', visa_fechaalta_date, fecha_foto) AS antiguedad_meses_visa,
    DATE_DIFF('month', master_fechaalta_date, fecha_foto) AS antiguedad_meses_master

FROM base_agregaciones;

,Success


### Features historicas

2. Features Históricas (Ventana Temporal)El churn casi nunca es abrupto; suele haber entre 2 y 6 meses de abandono progresivo. Las siguientes operaciones deben aplicarse agrupando por numero_de_cliente y ordenando por foto_mes.Variables clave a trackear en el tiempo:Transaccionalidad: ctrx_quarter, cproductos, chomebanking_transacciones, cmobile_app_trx, cconsumos_total_tc.Montos: mcuentas_saldo, mpayroll, mrentabilidad, mconsumototal_tc.Transformaciones temporales recomendadas:Deltas y Derivadas (Diferencias Absolutas):$\Delta_1 = X_t - X_{t-1}$ (pérdida mensual directa)$\Delta_2 = X_t - X_{t-2}$$\Delta_3 = X_t - X_{t-3}$$\Delta_6 = X_t - X_{t-6}$Ratios de Variación (Cambios Relativos):$Ratio_1 = \frac{X_t + 1}{X_{t-1} + 1}$$Ratio_3 = \frac{X_t + 1}{X_{t-3} + 1}$(Si el ratio de transacciones o saldo cae fuertemente hacia 0, la probabilidad de churn se dispara).Medias Móviles y Tendencias (Rolling Windows de 3 y 6 meses):mean_3m, mean_6mratio_actual_vs_media3m = X_t / (mean_3m + 0.01)tendencia_lineal: pendiente (slope) de los últimos 3 o 6 meses para variables como saldo, rentabilidad o transacciones.Volatilidad y Estabilidad:std_3m, std_6m (clientes con caída a desviación cero en transacciones indican inactividad total).Mínimos, Máximos y Distancia a Picos:max_6m, min_6mdistancia_al_pico = X_t / (max_6m + 0.01) (indica qué tan lejos está el cliente de su nivel máximo de transacciones o saldo de los últimos 6 meses).3. Señales Directas de Abandono (Flags y Conteos de Eventos)Cesación de Sueldo (mpayroll / cpayroll_trx):Flag si mpayroll_t == 0 y mpayroll_{t-1} > 0 (cambio de banco para cobro de haberes).Vaciado de Cuentas:Flag si mcuentas_saldo_t <= 0 habiendo tenido saldo positivo en $t-1$ o $t-2$.Inactividad de Tarjetas:Meses consecutivos sin consumos en Visa y Master: meses_sin_consumo_tc.Fuga de Débitos Automáticos:$\Delta(ccuenta\_debitos\_automaticos + ctarjeta\_visa\_debitos\_automaticos + ctarjeta\_master\_debitos\_automaticos)$ a 3 meses.Mora y Cobranzas:Flag de cambio en Master_status o Visa_status.Master_delinquency_t - Master_delinquency_{t-1} (incremento de días de atraso).

In [8]:
%%sql
CREATE OR REPLACE TABLE competencia_01_hist AS
WITH base_lags AS (
    SELECT
        *,

        -- Lags de orden 1 (t-1)
        LAG(ctrx_quarter, 1)               OVER w AS ctrx_quarter_lag1,
        LAG(cproductos, 1)                 OVER w AS cproductos_lag1,
        LAG(chomebanking_transacciones, 1) OVER w AS chb_trx_lag1,
        LAG(cmobile_app_trx, 1)            OVER w AS cmobile_trx_lag1,
        LAG(cconsumos_total_tc, 1)         OVER w AS cconsumos_tc_lag1,
        LAG(mcuentas_saldo, 1)             OVER w AS mcuentas_saldo_lag1,
        LAG(mpayroll, 1)                   OVER w AS mpayroll_lag1,
        LAG(mrentabilidad, 1)              OVER w AS mrentabilidad_lag1,
        LAG(mconsumototal_tc, 1)           OVER w AS mconsumototal_tc_lag1,

        -- Lags de orden 2 (t-2) [Solo útiles si tu base incluye meses previos a marzo]
        LAG(ctrx_quarter, 2)               OVER w AS ctrx_quarter_lag2,
        LAG(cproductos, 2)                 OVER w AS cproductos_lag2,
        LAG(cconsumos_total_tc, 2)         OVER w AS cconsumos_tc_lag2,
        LAG(mcuentas_saldo, 2)             OVER w AS mcuentas_saldo_lag2,
        LAG(mpayroll, 2)                   OVER w AS mpayroll_lag2,
        LAG(mrentabilidad, 2)              OVER w AS mrentabilidad_lag2,
        LAG(mconsumototal_tc, 2)           OVER w AS mconsumototal_tc_lag2,

        -- Ventanas de 3 meses
        AVG(ctrx_quarter)                  OVER w3 AS ctrx_quarter_avg3m,
        STDDEV(ctrx_quarter)               OVER w3 AS ctrx_quarter_std3m,
        AVG(mcuentas_saldo)                OVER w3 AS mcuentas_saldo_avg3m,
        STDDEV(mcuentas_saldo)             OVER w3 AS mcuentas_saldo_std3m,
        AVG(mrentabilidad)                 OVER w3 AS mrentabilidad_avg3m,
        AVG(mconsumototal_tc)              OVER w3 AS mconsumototal_tc_avg3m

    FROM competencia_01_fe
    WINDOW 
        w  AS (PARTITION BY numero_de_cliente ORDER BY foto_mes),
        w3 AS (PARTITION BY numero_de_cliente ORDER BY foto_mes ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
)
SELECT
    *,

    -- Deltas absolutos (t - t_lag)
    (ctrx_quarter - ctrx_quarter_lag1)           AS ctrx_quarter_delta1,
    (ctrx_quarter - ctrx_quarter_lag2)           AS ctrx_quarter_delta2,

    (cproductos - cproductos_lag1)               AS cproductos_delta1,
    (cproductos - cproductos_lag2)               AS cproductos_delta2,

    (mcuentas_saldo - mcuentas_saldo_lag1)       AS mcuentas_saldo_delta1,
    (mcuentas_saldo - mcuentas_saldo_lag2)       AS mcuentas_saldo_delta2,

    (mrentabilidad - mrentabilidad_lag1)         AS mrentabilidad_delta1,
    (mrentabilidad - mrentabilidad_lag2)         AS mrentabilidad_delta2,

    (mconsumototal_tc - mconsumototal_tc_lag1)   AS mconsumototal_tc_delta1,
    (mconsumototal_tc - mconsumototal_tc_lag2)   AS mconsumototal_tc_delta2,

    -- Ratios relativos con suavizado aditivo (evita división por 0)
    ROUND((ctrx_quarter + 1.0) / NULLIF(ctrx_quarter_lag1 + 1.0, 0), 4)       AS ctrx_quarter_ratio1,
    ROUND((ctrx_quarter + 1.0) / NULLIF(ctrx_quarter_lag2 + 1.0, 0), 4)       AS ctrx_quarter_ratio2,

    ROUND((cconsumos_total_tc + 1.0) / NULLIF(cconsumos_tc_lag1 + 1.0, 0), 4) AS cconsumos_tc_ratio1,
    ROUND((cconsumos_total_tc + 1.0) / NULLIF(cconsumos_tc_lag2 + 1.0, 0), 4) AS cconsumos_tc_ratio2,

    -- Ratios contra promedio móvil
    ROUND(ctrx_quarter / NULLIF(ctrx_quarter_avg3m, 0), 4)                    AS ctrx_quarter_ratio_avg3m,
    ROUND(mcuentas_saldo / NULLIF(mcuentas_saldo_avg3m, 0), 4)                AS mcuentas_saldo_ratio_avg3m,

    -- Flags binarios directos de alerta de churn
    CASE 
        WHEN COALESCE(mpayroll, 0) = 0 AND COALESCE(mpayroll_lag1, 0) > 0 THEN 1 
        ELSE 0 
    END AS flag_cesacion_payroll,

    CASE 
        WHEN COALESCE(mcuentas_saldo, 0) <= 0 AND COALESCE(mcuentas_saldo_lag1, 0) > 0 THEN 1 
        ELSE 0 
    END AS flag_vaciado_cuentas,

    CASE 
        WHEN COALESCE(cconsumos_total_tc, 0) = 0 AND COALESCE(cconsumos_tc_lag1, 0) > 0 THEN 1 
        ELSE 0 
    END AS flag_abandono_consumo_tc

FROM base_lags;

,Success


### Guardado

In [9]:
output_path = f"{dataset_path}competencia_01_fev2.parquet"

In [10]:
output_path

'/Users/carof/Downloads/DENICOLAY/data/competencia_01_fev2.parquet'

In [11]:
%%sql
COPY competencia_01_hist TO '{{output_path}}' (FORMAT PARQUET, COMPRESSION ZSTD);

,Success


In [12]:
output_path = f"{dataset_path}competencia_01_fev2.csv"

In [13]:
%%sql
COPY competencia_01_hist TO "{{output_path}}" (HEADER, DELIMITER ',');

,Success
